In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

BASE = "/kaggle/input/notebooks/krishnaiiith/05-proposed-model"

X = np.load(BASE + "/multimodal_features.npy")
y = np.load(BASE + "/multimodal_labels.npy")

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

Feature shape: (11902, 3077)
Label shape: (11902,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (9521, 3077)
Test shape: (2381, 3077)


In [4]:
print("Train class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))

Train class distribution: [8328 1193]
Test class distribution: [2083  298]


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(model, name):

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities (some models may not support predict_proba)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:,1]
    else:
        y_prob = model.decision_function(X_test)

    roc = roc_auc_score(y_test, y_prob)

    print("\n" + "="*50)
    print(name)
    print("="*50)
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc)

    return roc

In [8]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

roc_lr = evaluate_model(lr, "Logistic Regression")


Logistic Regression
              precision    recall  f1-score   support

           0       0.92      0.87      0.89      2083
           1       0.33      0.44      0.38       298

    accuracy                           0.82      2381
   macro avg       0.62      0.66      0.63      2381
weighted avg       0.84      0.82      0.83      2381

ROC-AUC: 0.7334212400158522


In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

roc_rf = evaluate_model(rf, "Random Forest")


Random Forest
              precision    recall  f1-score   support

           0       0.88      0.99      0.93      2083
           1       0.54      0.05      0.09       298

    accuracy                           0.88      2381
   macro avg       0.71      0.52      0.51      2381
weighted avg       0.84      0.88      0.83      2381

ROC-AUC: 0.781347565946122


In [10]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

roc_xgb = evaluate_model(xgb, "XGBoost")


XGBoost
              precision    recall  f1-score   support

           0       0.92      0.94      0.93      2083
           1       0.52      0.43      0.47       298

    accuracy                           0.88      2381
   macro avg       0.72      0.69      0.70      2381
weighted avg       0.87      0.88      0.87      2381

ROC-AUC: 0.8257361446287782


In [11]:
mlp = MLPClassifier(
    hidden_layer_sizes=(512,128),
    max_iter=200,
    batch_size=64,
    learning_rate_init=0.001,
    early_stopping=True,
    random_state=42
)

roc_mlp = evaluate_model(mlp, "MLP Neural Network")


MLP Neural Network
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      2083
           1       0.68      0.27      0.39       298

    accuracy                           0.89      2381
   macro avg       0.79      0.63      0.66      2381
weighted avg       0.88      0.89      0.87      2381

ROC-AUC: 0.7948195201165072


In [12]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "MLP"
    ],
    "ROC-AUC": [
        roc_lr,
        roc_rf,
        roc_xgb,
        roc_mlp
    ]
})

results

,Model,ROC-AUC
0,Logistic Regression,0.733421
1,Random Forest,0.781348
2,XGBoost,0.825736
3,MLP,0.794820
